# Project - Airline AI Assistant

We'll now bring together what we've learned to make an AI Customer Support assistant for an Airline

In [ ]:
# imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sqlite3

In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

# Initialization
load_dotenv(override=True)

# 1. Swap to Groq API Key
groq_api_key = os.getenv('GROQ_API_KEY')
if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:8]}")
else:
    print("Groq API Key not set - check your .env file!")
    
# 2. Swap to a Groq-supported model
MODEL = "llama-3.3-70b-versatile"

# 3. Point the OpenAI client to Groq's servers
openai = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=groq_api_key
)

DB = "prices.db"

In [ ]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [ ]:
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

In [ ]:
get_ticket_price("Paris")

In [ ]:
price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}
tools = [{"type": "function", "function": price_function}]
tools

In [ ]:

def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

gr.ChatInterface(fn=chat, type="messages").launch()

In [ ]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content

In [ ]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

## A bit more about what Gradio actually does:

1. Gradio constructs a frontend Svelte app based on our Python description of the UI
2. Gradio starts a server built upon the Starlette web framework listening on a free port that serves this React app
3. Gradio creates backend routes for our callbacks, like chat(), which calls our functions

And of course when Gradio generates the frontend app, it ensures that the the Submit button calls the right backend route.

That's it!

It's simple, and it has a result that feels magical.

# Let's go multi-modal!!

We can use DALL-E-3, the image generation model behind GPT-4o, to make us some images

Let's put this in a function called artist.

### Price alert: each time I generate an image it costs about 4 cents - don't go crazy with images!

In [ ]:
import os
import base64
from io import BytesIO
from PIL import Image
from openai import OpenAI

In [ ]:
openai_image_client = OpenAI()
def artist(city):
    image_response = openai.images.generate(
            model="dall-e-3",
            prompt=f"An image representing a vacation in {city}, showing tourist spots and everything unique about {city}, in a vibrant pop-art style",
            size="1024x1024",
            n=1,
            response_format="b64_json",
        )
    image_base64 = image_response.data[0].b64_json
    image_data = base64.b64decode(image_base64)
    return Image.open(BytesIO(image_data))

In [ ]:
image = artist("New York City")
display(image)

In [ ]:
import os
import json
import sqlite3
import gradio as gr
from io import BytesIO
from PIL import Image
from openai import OpenAI
from google import genai
from google.genai import types
from dotenv import load_dotenv

# ==========================================
# 1. Initialization & Dual-Client Setup
# ==========================================
load_dotenv(override=True)

# Client A: Groq (For lightning-fast text & tool calling)
groq_api_key = os.getenv('GROQ_API_KEY')
# If it EVER fails tool calling again, you can change this to "llama3-groq-70b-8192-tool-use-preview"
MODEL = "llama-3.3-70b-versatile" 
groq_client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=groq_api_key
)

# Client B: Google Gemini (For free Imagen 3 Image Generation)
gemini_api_key = os.getenv('GEMINI_API_KEY')
if not gemini_api_key:
    print("WARNING: Gemini API Key not found! Check your .env file.")
gemini_client = genai.Client(api_key=gemini_api_key)

# ==========================================
# 2. Database & Functions
# ==========================================
DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
    conn.commit()

def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

def set_ticket_price(city, price):
    print(f"DATABASE TOOL CALLED: Setting price for {city} to {price}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()
    return f"Successfully updated the price for {city} to ${price}"

# Seed the database
ticket_prices = {"london":799, "paris": 899, "tokyo": 1420, "sydney": 2999}
for city, price in ticket_prices.items():
    set_ticket_price(city, price)

# ==========================================
# 3. Image Generation Function (Gemini / Imagen 3)
# ==========================================
def artist(city):
    print(f"IMAGE TOOL CALLED: Drawing {city} using Gemini Imagen 3", flush=True)
    prompt = f"An image representing a vacation in {city}, showing tourist spots and everything unique about {city}, in a vibrant pop-art style"
    
    # Call the Imagen 3 model via the free Gemini API
    response = gemini_client.models.generate_images(
        model='imagen-3.0-generate-001', # <--- Fixed model ID!
        prompt=prompt,
        config=types.GenerateImagesConfig(
            number_of_images=1,
            aspect_ratio="1:1"
        )
    )
    
    # Extract the raw image bytes and convert to a PIL Image
    generated_image = response.generated_images[0]
    return Image.open(BytesIO(generated_image.image.image_bytes))
# ==========================================
# 4. Define the Tools for the AI
# ==========================================
get_price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

set_price_function = {
    "name": "set_ticket_price",
    "description": "Update or set the price of a return ticket to a specific city in the database.",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {
                "type": "string",
                "description": "The city to update the price for",
            },
            "price": {
                "type": "number",
                "description": "The new price in dollars",
            },
        },
        "required": ["city", "price"],
        "additionalProperties": False
    }
}

tools = [
    {"type": "function", "function": get_price_function},
    {"type": "function", "function": set_price_function}
]

# ==========================================
# 5. Handle the Tool Calls
# ==========================================
def handle_tool_calls(message):
    responses = []
    if not message.tool_calls:
        return responses
        
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
            
        elif tool_call.function.name == "set_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('city')
            price = arguments.get('price')
            
            result_message = set_ticket_price(city, price) 
            
            responses.append({
                "role": "tool",
                "content": result_message,
                "tool_call_id": tool_call.id
            })
            
    return responses

# ==========================================
# 6. Chat Function & Gradio UI
# ==========================================
system_message = """
You are a helpful assistant for an Airline called FlightAI.
You can check ticket prices for users. 
If an administrator asks you to update or set a price, use your tool to do so.
Always be accurate and polite.

IMPORTANT STRICT RULE: 
When calling a tool or function, you must output ONLY perfectly formatted JSON. 
Do not use XML tags, <function> tags, or any other custom formatting.
"""

def chat(message, history):
    processed_history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + processed_history + [{"role": "user", "content": message}]
    
    response = groq_client.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    while response.choices[0].finish_reason == "tool_calls":
        assistant_message = response.choices[0].message
        messages.append(assistant_message)
        
        tool_responses = handle_tool_calls(assistant_message)
        messages.extend(tool_responses)
        
        response = groq_client.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content

# Launch the Gradio Interface
gr.ChatInterface(fn=chat, type="messages").launch(share=False)

In [ ]:
import os
import json
import sqlite3
import gradio as gr
from io import BytesIO
from PIL import Image
from openai import OpenAI
from google import genai
from google.genai import types
from dotenv import load_dotenv
from IPython.display import display

# ==========================================
# 1. Initialization & Setup
# ==========================================
load_dotenv(override=True)

# Groq Setup
groq_api_key = os.getenv('GROQ_API_KEY')
MODEL = "llama-3.3-70b-versatile" 
groq_client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=groq_api_key
)

# Gemini Setup
gemini_api_key = os.getenv('GEMINI_API_KEY')
gemini_client = genai.Client(api_key=gemini_api_key)

# ==========================================
# 2. Database & Functions
# ==========================================
DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
    conn.commit()

def get_ticket_price(city):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

def set_ticket_price(city, price):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()
    return f"Successfully updated the price for {city} to ${price}"

# Seed Database
ticket_prices = {"london":799, "paris": 899, "tokyo": 1420, "sydney": 2999}
for city, price in ticket_prices.items():
    set_ticket_price(city, price)

# ==========================================
# 3. Image Generation Function (Updated to generate_content)
# ==========================================
def artist(city):
    print(f"🎨 Generating pop-art image for {city} via Gemini...", flush=True)
    prompt = f"An image representing a vacation in {city}, showing tourist spots and everything unique about {city}, in a vibrant pop-art style"
    
    # NEW SDK METHOD: Using generate_content for the Gemini image models
    response = gemini_client.models.generate_content(
        model='gemini-2.5-flash-image',
        contents=prompt,
        config=types.GenerateContentConfig(
            response_modalities=["IMAGE"],
            image_config=types.ImageConfig(
                aspect_ratio="1:1"
            )
        )
    )
    
    # Extract the generated image from the response payload
    for part in response.parts:
        if part.inline_data:
            return Image.open(BytesIO(part.inline_data.data))

# ==========================================
# 4. Generate Image Test 
# ==========================================
try:
    london_image = artist("London")
    display(london_image)
    print("✅ Image generated successfully!")
except Exception as e:
    print(f"❌ Image generation failed: {e}")

# ==========================================
# 5. Define Tools & Handle Calls
# ==========================================
get_price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {"destination_city": {"type": "string"}},
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

set_price_function = {
    "name": "set_ticket_price",
    "description": "Update or set the price of a return ticket to a specific city in the database.",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {"type": "string"},
            "price": {"type": "number"}
        },
        "required": ["city", "price"],
        "additionalProperties": False
    }
}

tools = [
    {"type": "function", "function": get_price_function},
    {"type": "function", "function": set_price_function}
]

def handle_tool_calls(message):
    responses = []
    if not message.tool_calls:
        return responses
        
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            args = json.loads(tool_call.function.arguments)
            responses.append({
                "role": "tool",
                "content": get_ticket_price(args.get('destination_city')),
                "tool_call_id": tool_call.id
            })
            
        elif tool_call.function.name == "set_ticket_price":
            args = json.loads(tool_call.function.arguments)
            responses.append({
                "role": "tool",
                "content": set_ticket_price(args.get('city'), args.get('price')),
                "tool_call_id": tool_call.id
            })
    return responses

# ==========================================
# 6. Chat Function & Gradio UI
# ==========================================
system_message = """
You are a helpful assistant for an Airline called FlightAI.
You can check ticket prices for users. 
If an administrator asks you to update or set a price, use your tool to do so.
Always be accurate and polite.

IMPORTANT STRICT RULE: 
When calling a tool or function, you must output ONLY perfectly formatted JSON. 
Do not use XML tags, <function> tags, or any other custom formatting.
"""

def chat(message, history):
    processed_history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + processed_history + [{"role": "user", "content": message}]
    
    response = groq_client.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    while response.choices[0].finish_reason == "tool_calls":
        assistant_message = response.choices[0].message
        messages.append(assistant_message)
        
        tool_responses = handle_tool_calls(assistant_message)
        messages.extend(tool_responses)
        
        response = groq_client.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content

# Launch the UI
gr.ChatInterface(fn=chat, type="messages").launch(share=False)

In [ ]:
def talker(message):
    response = openai.audio.speech.create(
      model="gpt-4o-mini-tts",
      voice="onyx",    # Also, try replacing onyx with alloy or coral
      input=message
    )
    return response.content

## Let's bring this home:

1. A multi-modal AI assistant with image and audio generation
2. Tool callling with database lookup
3. A step towards an Agentic workflow


In [ ]:
def chat(history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    cities = []
    image = None

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses, cities = handle_tool_calls_and_return_cities(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    reply = response.choices[0].message.content
    history += [{"role":"assistant", "content":reply}]

    voice = talker(reply)

    if cities:
        image = artist(cities[0])
    
    return history, voice, image


In [ ]:
def handle_tool_calls_and_return_cities(message):
    responses = []
    cities = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            cities.append(city)
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses, cities

## The 3 types of Gradio UI

`gr.Interface` is for standard, simple UIs

`gr.ChatInterface` is for standard ChatBot UIs

`gr.Blocks` is for custom UIs where you control the components and the callbacks

In [ ]:
# Callbacks (along with the chat() function above)

def put_message_in_chatbot(message, history):
        return "", history + [{"role":"user", "content":message}]

# UI definition

with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500, type="messages")
        image_output = gr.Image(height=500, interactive=False)
    with gr.Row():
        audio_output = gr.Audio(autoplay=True)
    with gr.Row():
        message = gr.Textbox(label="Chat with our AI Assistant:")

# Hooking up events to callbacks

    message.submit(put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]).then(
        chat, inputs=chatbot, outputs=[chatbot, audio_output, image_output]
    )

ui.launch(inbrowser=True, auth=("ed", "bananas"))

# Exercises and Business Applications

Add in more tools - perhaps to simulate actually booking a flight. A student has done this and provided their example in the community contributions folder.

Next: take this and apply it to your business. Make a multi-modal AI assistant with tools that could carry out an activity for your work. A customer support assistant? New employee onboarding assistant? So many possibilities! Also, see the week2 end of week Exercise in the separate Notebook.

In [ ]:
!pip install gTTS

In [ ]:
import os
import io
import gradio as gr
from PIL import Image
from google import genai
from google.genai import types
from dotenv import load_dotenv
from gtts import gTTS  # NEW: Bulletproof & free Text-to-Speech!

# ==========================================
# 1. Setup & Initialization
# ==========================================
load_dotenv(override=True)
gemini_api_key = os.getenv('GEMINI_API_KEY')
client = genai.Client(api_key=gemini_api_key)

# ==========================================
# 2. Callbacks & Logic
# ==========================================
def put_message_in_chatbot(message, history):
    # Appends the user's message to the history UI instantly
    return "", history + [{"role": "user", "content": message}]

def chat(history):
    # Grab the actual text the user just submitted
    user_message = history[-1]["content"]
    
    # --- A. Generate Text ---
    try:
        # Swapped to 2.0-flash to bypass 503 high-traffic errors
        text_response = client.models.generate_content(
            model='gemini-2.0-flash',
            contents=user_message
        )
        ai_text = text_response.text
    except Exception as e:
        ai_text = f"Oops, text generation failed: {e}"
        
    # Append the AI's response to the chat history
    history.append({"role": "assistant", "content": ai_text})
    
    # --- B. Generate Audio (gTTS) ---
    generated_audio = None
    try:
        # Using Google TTS natively instead of a broken API endpoint
        tts = gTTS(ai_text, lang='en')
        audio_path = "output_audio.mp3"
        tts.save(audio_path)
        generated_audio = audio_path
    except Exception as e:
        print(f"Audio Error (Skipping): {e}")
        
    # --- C. Generate Image (Imagen 3 API updated) ---
    generated_image = None
    try:
        # Updated to the new 002 version!
        image_response = client.models.generate_images(
            model='imagen-3.0-generate-002',
            prompt=user_message + ", vibrant pop art style, high quality",
            config=types.GenerateImagesConfig(
                number_of_images=1,
                aspect_ratio="1:1"
            )
        )
        # Extract the raw image bytes and convert to PIL Image for Gradio
        img_bytes = image_response.generated_images[0].image.image_bytes
        generated_image = Image.open(io.BytesIO(img_bytes))
    except Exception as e:
        print(f"Image Error (Skipping): {e}")

    # Must return EXACTLY 3 items to match your UI outputs list below
    return history, generated_audio, generated_image

# ==========================================
# 3. UI Definition & Routing
# ==========================================
with gr.Blocks(theme=gr.themes.Soft()) as ui:
    gr.Markdown("# ✈️ FlightAI Multimodal Assistant")
    
    with gr.Row():
        chatbot = gr.Chatbot(height=450, type="messages", label="Conversation")
        image_output = gr.Image(height=450, interactive=False, label="Generated Scene")
        
    with gr.Row():
        audio_output = gr.Audio(autoplay=True, label="Voice Output")
        
    with gr.Row():
        message = gr.Textbox(label="Chat with our AI Assistant (e.g., 'Describe a vacation in London')")
        
    # Hooking up events to callbacks
    message.submit(
        put_message_in_chatbot, 
        inputs=[message, chatbot], 
        outputs=[message, chatbot]
    ).then(
        chat, 
        inputs=chatbot, 
        outputs=[chatbot, audio_output, image_output]
    )

# ==========================================
# 4. Launch with Auth!
# ==========================================
ui.launch(auth=("Shaheed", "1234"), share=False)

In [ ]:
import os
import io
import gradio as gr
from PIL import Image
from google import genai
from google.genai import types
from dotenv import load_dotenv
from gtts import gTTS

# ==========================================
# 1. Setup & Initialization
# ==========================================
load_dotenv(override=True)
gemini_api_key = os.getenv('GEMINI_API_KEY')
client = genai.Client(api_key=gemini_api_key)

# ==========================================
# 2. Callbacks & Logic
# ==========================================
def put_message_in_chatbot(message, history):
    return "", history + [{"role": "user", "content": message}]

def chat(history):
    user_message = history[-1]["content"]
    is_error = False
    
    # --- A. Generate Text ---
    try:
        # Swapped to 1.5-flash for MUCH higher free tier rate limits!
        text_response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=user_message
        )
        ai_text = text_response.text
    except Exception as e:
        ai_text = f"Oops, API limit reached. Please wait 60 seconds and try again. \n\nError details: {e}"
        is_error = True
        
    history.append({"role": "assistant", "content": ai_text})
    
    # --- B. Generate Audio (Fixed to not read errors!) ---
    generated_audio = None
    try:
        # If there's an error, say something short instead of the JSON log
        speech_text = "I'm sorry, I am experiencing high traffic right now. Please wait a minute and try again." if is_error else ai_text
        
        tts = gTTS(speech_text, lang='en')
        audio_path = "output_audio.mp3"
        tts.save(audio_path)
        generated_audio = audio_path
    except Exception as e:
        print(f"Audio Error (Skipping): {e}")
        
    # --- C. Generate Image ---
    generated_image = None
    # Only try to generate an image if we aren't already rate limited
    if not is_error:
        try:
            image_response = client.models.generate_images(
                model='imagen-3.0-generate-002',
                prompt=user_message + ", vibrant pop art style, high quality",
                config=types.GenerateImagesConfig(
                    number_of_images=1,
                    aspect_ratio="1:1"
                )
            )
            img_bytes = image_response.generated_images[0].image.image_bytes
            generated_image = Image.open(io.BytesIO(img_bytes))
        except Exception as e:
            print(f"Image Error (Skipping): {e}")

    return history, generated_audio, generated_image

# ==========================================
# 3. UI Definition & Routing
# ==========================================
with gr.Blocks(theme=gr.themes.Soft()) as ui:
    gr.Markdown("# ✈️ FlightAI Multimodal Assistant")
    
    with gr.Row():
        chatbot = gr.Chatbot(height=450, type="messages", label="Conversation")
        image_output = gr.Image(height=450, interactive=False, label="Generated Scene")
        
    with gr.Row():
        audio_output = gr.Audio(autoplay=True, label="Voice Output")
        
    with gr.Row():
        message = gr.Textbox(label="Chat with our AI Assistant (e.g., 'Describe a vacation in London')")
        
    message.submit(
        put_message_in_chatbot, 
        inputs=[message, chatbot], 
        outputs=[message, chatbot]
    ).then(
        chat, 
        inputs=chatbot, 
        outputs=[chatbot, audio_output, image_output]
    )

# ==========================================
# 4. Launch!
# ==========================================
ui.launch(auth=("Shaheed", "1234"), share=False)

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a HUGE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>